# Adversarial Testing (V2 Defense Model)

اختبار رسائل الهجوم المولّدة ضد موديل الدفاع المدرّب (V2).

> **ملاحظة:** هذا النوتبوك يعمل **بعد** ما الهجوم والدفاع كلهم جاهزين.

---


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent.parent))
sys.path.insert(0, str(Path().resolve().parent.parent / 'attack' / 'src'))

import pandas as pd
from generator import PhishingGenerator
from mutator import AdversarialMutator
from adversarial.src.evaluator import AdversarialEvaluator

# V2 defense model path
ROOT = Path().resolve().parent.parent
MODEL_V2 = ROOT / 'model_defenseV2' / 'models' / 'defence_model_v2.pkl'
print(f'Using defense model: {MODEL_V2}')


## 1. توليد رسائل تصيد جديدة


In [ ]:
df = pd.read_csv('../../data/processed/final_dataset_v2.csv').drop_duplicates(subset='text')
gen = PhishingGenerator(seed=99)  # different seed for fresh messages
gen.fit(df[df['Label']==0]['text'].tolist())

phishing_msgs = [m['text'] for m in gen.generate(count=50)]
print(f'Generated {len(phishing_msgs)} fresh phishing messages')


## 2. اختبار ضد موديل الدفاع (V2)


In [ ]:
evl = AdversarialEvaluator(model_path=str(MODEL_V2))
report = evl.evaluate(phishing_msgs)

print(f'=== Defense Model Results (V2) ===')
print(f'  Tested:         {report["total"]}')
print(f'  Detected:       {report["detected"]}')
print(f'  Evaded:         {report["evaded"]}')
print(f'  Detection rate: {report["detection_rate"]*100:.1f}%')
print(f'  Evasion rate:   {report["evasion_rate"]*100:.1f}%')

if report['evaded_messages']:
    print(f'\n🚨 Evaded ({len(report["evaded_messages"])}):')
    for e in report['evaded_messages'][:5]:
        print(f'  → {e["text"][:80]}...')
else:
    print('\n✅ No evasions!')


## 3. Adversarial Mutations


In [ ]:
mut = AdversarialMutator(seed=99)
mut_results = mut.mutate_batch(phishing_msgs[:20], mutations_per_text=2)

print(f'Mutated {len(mut_results)} messages')
for r in mut_results[:3]:
    if r['changed']:
        print(f'\n  Mutations: {r["mutations_applied"]}')
        print(f'  Before: {r["original"][:70]}...')
        print(f'  After:  {r["mutated"][:70]}...')


In [ ]:
mut_report = evl.evaluate_mutations(mut_results)
print(f'=== Mutation Effectiveness ===')
print(f'  Total evasions: {mut_report["total_evasions"]}/{mut_report["total_tested"]}')
print(f'  Evasion rate:   {mut_report["evasion_rate"]*100:.1f}%')

for m, s in mut_report['per_mutation_stats'].items():
    print(f'    {m:20s}: {s["evasions"]}/{s["total"]} ({s["evasion_rate"]*100:.0f}%)')


## ✅ النتائج

هذه النتائج تُستخدم لتحسين كل من:
- **الهجوم:** تطوير تقنيات تهرّب أقوى
- **الدفاع:** سد الثغرات المكتشفة

```
هجوم ←→ دفاع = تحسين مستمر
```
